# Preparation of a SILVA 138.2 V3–V4 taxonomic classifier

This notebook prepares a **QIIME 2 Naive Bayes taxonomic classifier** for the V3–V4 region of the 16S rRNA gene using **SILVA 138.2 SSU Ref NR99** and the primer pair **341F/806R**.

The workflow follows the same general sequence as the original notebook:

1. configure paths and execution parameters;
2. obtain and import SILVA 138.2 reference sequences;
3. convert RNA reference sequences to DNA;
4. obtain and import SILVA taxonomy resources;
5. generate fixed-rank taxonomy with RESCRIPt;
6. extract the V3–V4 region in silico;
7. train Naive Bayes classifiers;
8. classify the reference sequences;
9. compare expected and observed taxonomy.

The notebook keeps the QIIME 2 commands explicit through `!qiime`, making each step easy to inspect and reproduce.

> **Methodological note:** the final comparison classifies the same reference sequences used to train the classifier. Therefore, it should be interpreted as an internal consistency check rather than as an independent estimate of classifier performance.

### Main references

- Quast C, Pruesse E, Yilmaz P, et al. *The SILVA ribosomal RNA gene database project*. Nucleic Acids Research. 2013.
- Yilmaz P, Parfrey LW, Yarza P, et al. *The SILVA and “All-species Living Tree Project (LTP)” taxonomic frameworks*. Nucleic Acids Research. 2014.
- Bokulich NA, Kaehler BD, Rideout JR, et al. *Optimizing taxonomic classification of marker-gene amplicon sequences with QIIME 2's q2-feature-classifier plugin*. Microbiome. 2018.
- Robeson MS II, O'Rourke DR, Kaehler BD, et al. *RESCRIPt: Reproducible sequence taxonomy reference database management*. PLoS Computational Biology. 2021.


## 1. Configuration

The configuration is divided into two categories:

- **Project parameters:** values that define the classifier itself and should remain the same across computational environments.
- **Environment parameters:** paths and resource settings that depend on the machine where the notebook is executed.

This separation avoids embedding workstation- or server-specific paths in the scientific workflow.


In [ ]:
from pathlib import Path
import os

# ============================================================
# PROJECT PARAMETERS
# ============================================================

SILVA_VERSION = "138.2"
SILVA_TARGET = "SSURef_NR99"

FORWARD_PRIMER = "CCTACGGGRSGCAGCAG"
REVERSE_PRIMER = "GGACTACHVGGGTWTCTAAT"

MIN_LENGTH = 350
MAX_LENGTH = 550

# Parallelization
N_JOBS_EXTRACT = 8
N_JOBS_CLASSIFY = 4

# Optional conservative batch size for classify-sklearn
READS_PER_BATCH = 1000


# ============================================================
# ENVIRONMENT PARAMETERS
# ============================================================

# Directory where project files and classifier artifacts are stored.
# Change this path according to the execution environment.
PROJECT_DIR = Path("/path/to/silva-138.2-dev").resolve()

# Directory with ample disk space for temporary files.
# This should point to a filesystem with sufficient free space.
TEMP_DIR = Path("/path/to/large/tmp").resolve()

# Separate Joblib temporary directory used by classify-sklearn.
JOBLIB_TEMP_DIR = TEMP_DIR / "joblib"

# Optional QIIME 2 cache directory.
QIIME_CACHE = TEMP_DIR / "qiime2-cache"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)
TEMP_DIR.mkdir(parents=True, exist_ok=True)
JOBLIB_TEMP_DIR.mkdir(parents=True, exist_ok=True)
QIIME_CACHE.mkdir(parents=True, exist_ok=True)

os.environ["TMPDIR"] = str(TEMP_DIR)
os.environ["TMP"] = str(TEMP_DIR)
os.environ["TEMP"] = str(TEMP_DIR)
os.environ["JOBLIB_TEMP_FOLDER"] = str(JOBLIB_TEMP_DIR)

os.chdir(PROJECT_DIR)

print("PROJECT_DIR        :", PROJECT_DIR)
print("TEMP_DIR           :", TEMP_DIR)
print("JOBLIB_TEMP_FOLDER :", JOBLIB_TEMP_DIR)
print("QIIME_CACHE        :", QIIME_CACHE)


### 1.1 Check the execution environment

Large QIIME 2 classifiers may require substantial temporary storage during training, loading, and parallel classification. In particular:

- QIIME 2 may extract artifacts into the temporary directory;
- Joblib may create temporary memory-mapped arrays when `classify-sklearn` uses multiple workers;
- the cache may also consume additional storage.

Before starting the pipeline, confirm that the configured temporary filesystem has sufficient free space.


In [ ]:
!echo "TMPDIR=$TMPDIR"
!echo "JOBLIB_TEMP_FOLDER=$JOBLIB_TEMP_FOLDER"
!python -c "import tempfile; print('Python temp directory:', tempfile.gettempdir())"

!df -h "$PROJECT_DIR" "$TEMP_DIR"


In [ ]:
!qiime --version
!python -c "import sklearn, joblib; print('scikit-learn:', sklearn.__version__); print('joblib:', joblib.__version__)"


## 2. SILVA 138.2 SSU Ref NR99 reference sequences

SILVA provides curated ribosomal RNA reference datasets. The **SSU Ref NR99** dataset is a non-redundant small-subunit reference collection clustered at approximately 99% identity and is suitable for taxonomic reference workflows.

The sequence export used here is:

`SILVA_138.2_SSURef_NR99_tax_silva.fasta.gz`

If the automatic RESCRIPt download is unreliable in the execution environment, the reference FASTA can be downloaded manually and imported into QIIME 2.


In [ ]:
SILVA_FASTA = f"SILVA_{SILVA_VERSION}_SSURef_NR99_tax_silva.fasta.gz"

print(SILVA_FASTA)


In [ ]:
URL=f"https://www.arb-silva.de/fileadmin/silva_databases/release_138_2/Exports/{SILVA_FASTA}"
!wget --continue \
      --tries=20 \
      --timeout=120 \
      --read-timeout=120 \
      --retry-connrefused \
      --waitretry=10 \
      --output-document={SILVA_FASTA} \
      {URL}

### 2.1 Optional automatic retrieval with RESCRIPt

This is the preferred reproducible route when network connectivity is stable.


In [ ]:
# Uncomment to download directly through RESCRIPt.
#
# !qiime rescript get-silva-data \
#     --p-version {SILVA_VERSION} \
#     --p-target {SILVA_TARGET} \
#     --o-silva-sequences silva-{SILVA_VERSION}-ssu-nr99-rna-seqs.qza \
#     --o-silva-taxonomy silva-{SILVA_VERSION}-ssu-nr99-tax.qza \
#     --use-cache {QIIME_CACHE} \
#     --verbose


### 2.2 Manual-download fallback

If the reference FASTA was downloaded manually, verify that the gzip archive is intact before importing it.


In [ ]:
!ls -lh "$SILVA_FASTA"
!gzip -t "$SILVA_FASTA"


### 2.3 Import RNA reference sequences


In [ ]:
!qiime tools import \
    --type 'FeatureData[RNASequence]' \
    --input-path "$SILVA_FASTA" \
    --output-path silva-{SILVA_VERSION}-ssu-nr99-rna-seqs.qza


In [ ]:
!qiime tools peek silva-{SILVA_VERSION}-ssu-nr99-rna-seqs.qza

!qiime tools validate \
    silva-{SILVA_VERSION}-ssu-nr99-rna-seqs.qza


## 3. Convert SILVA RNA sequences to DNA

SILVA SSU reference exports contain RNA sequences. For primer-based extraction with `feature-classifier extract-reads`, the reference sequences are converted to DNA representation using RESCRIPt.


In [ ]:
!qiime rescript reverse-transcribe \
    --i-rna-sequences silva-{SILVA_VERSION}-ssu-nr99-rna-seqs.qza \
    --o-dna-sequences silva-{SILVA_VERSION}-ssu-nr99-dna-seqs.qza \
    --use-cache {QIIME_CACHE} \
    --verbose


In [ ]:
!qiime tools peek silva-{SILVA_VERSION}-ssu-nr99-dna-seqs.qza

!qiime tools validate \
    silva-{SILVA_VERSION}-ssu-nr99-dna-seqs.qza


## 4. SILVA taxonomy resources

RESCRIPt can reconstruct fixed-rank SILVA taxonomy using the SILVA taxonomy map, rank definition, and taxonomy tree.

The expected files for SILVA 138.2 are:

- `taxmap_slv_ssu_ref_nr_138.2.txt.gz`
- `tax_slv_ssu_138.2.txt.gz`
- `tax_slv_ssu_138.2.tre.gz`

These files should be placed in `PROJECT_DIR`.


In [ ]:
TAXMAP_GZ = f"taxmap_slv_ssu_ref_nr_{SILVA_VERSION}.txt.gz"
TAXONOMY_GZ = f"tax_slv_ssu_{SILVA_VERSION}.txt.gz"
TREE_GZ = f"tax_slv_ssu_{SILVA_VERSION}.tre.gz"

print(TAXMAP_GZ)
print(TAXONOMY_GZ)
print(TREE_GZ)


In [ ]:
BASE_URL="https://www.arb-silva.de/fileadmin/silva_databases/release_138_2/Exports/taxonomy"

!wget --continue --tries=20 \
    "{BASE_URL}/{TAXMAP_GZ}"

!wget --continue --tries=20 \
    "{BASE_URL}/{TAXONOMY_GZ}"

!wget --continue --tries=20 \
    "{BASE_URL}/{TREE_GZ}"

### 4.1 Decompress the taxonomy files

The `-k` option preserves the original compressed files.


In [ ]:
!gzip -dkf "$TAXMAP_GZ"
!gzip -dkf "$TAXONOMY_GZ"
!gzip -dkf "$TREE_GZ"


### 4.2 Import SILVA taxonomy components into QIIME 2


In [ ]:
!qiime tools import \
    --type 'FeatureData[SILVATaxidMap]' \
    --input-path taxmap_slv_ssu_ref_nr_{SILVA_VERSION}.txt \
    --output-path taxmap-slv-ssu-ref-nr-{SILVA_VERSION}.qza


In [ ]:
!qiime tools import \
    --type 'FeatureData[SILVATaxonomy]' \
    --input-path tax_slv_ssu_{SILVA_VERSION}.txt \
    --output-path tax-slv-ssu-{SILVA_VERSION}.qza


In [ ]:
!qiime tools import \
    --type 'Phylogeny[Rooted]' \
    --input-format NewickFormat \
    --input-path tax_slv_ssu_{SILVA_VERSION}.tre \
    --output-path tax-slv-ssu-{SILVA_VERSION}-tree.qza


## 5. Generate fixed-rank taxonomy

Two versions are generated:

1. taxonomy through the **genus** rank;
2. taxonomy including **species labels**.

The species-level taxonomy is retained for exploratory purposes. Species assignments from 16S amplicons should be interpreted cautiously because marker-region resolution and reference-label quality may limit discrimination among closely related taxa.


### 5.1 Taxonomy through genus


In [ ]:
!qiime rescript parse-silva-taxonomy \
    --i-taxonomy-tree tax-slv-ssu-{SILVA_VERSION}-tree.qza \
    --i-taxonomy-map taxmap-slv-ssu-ref-nr-{SILVA_VERSION}.qza \
    --i-taxonomy-ranks tax-slv-ssu-{SILVA_VERSION}.qza \
    --o-taxonomy silva-{SILVA_VERSION}-ssu-nr99-tax.qza \
    --use-cache {QIIME_CACHE} \
    --verbose


In [ ]:
!qiime tools peek silva-{SILVA_VERSION}-ssu-nr99-tax.qza

!qiime tools validate \
    silva-{SILVA_VERSION}-ssu-nr99-tax.qza


### 5.2 Taxonomy including species labels


In [ ]:
!qiime rescript parse-silva-taxonomy \
    --i-taxonomy-tree tax-slv-ssu-{SILVA_VERSION}-tree.qza \
    --i-taxonomy-map taxmap-slv-ssu-ref-nr-{SILVA_VERSION}.qza \
    --i-taxonomy-ranks tax-slv-ssu-{SILVA_VERSION}.qza \
    --p-include-species-labels \
    --o-taxonomy silva-{SILVA_VERSION}-ssu-nr99-species-tax.qza \
    --use-cache {QIIME_CACHE} \
    --verbose


In [ ]:
!qiime tools peek silva-{SILVA_VERSION}-ssu-nr99-species-tax.qza

!qiime tools validate \
    silva-{SILVA_VERSION}-ssu-nr99-species-tax.qza


## 6. Extract the V3–V4 region

The experimental amplicon targets the V3–V4 region of the 16S rRNA gene using primers 341F and 806R.

Primer sequences used in this project:

- **341F:** `CCTACGGGRSGCAGCAG`
- **806R:** `GGACTACHVGGGTWTCTAAT`

The reference sequences are therefore trimmed in silico so that the classifier is trained on the same marker region expected in the sequencing data.

The extraction accepts products between 350 and 550 bp to accommodate expected biological variation around the V3–V4 amplicon.


In [ ]:
!qiime feature-classifier extract-reads \
    --i-sequences silva-{SILVA_VERSION}-ssu-nr99-dna-seqs.qza \
    --p-f-primer {FORWARD_PRIMER} \
    --p-r-primer {REVERSE_PRIMER} \
    --p-read-orientation forward \
    --p-min-length {MIN_LENGTH} \
    --p-max-length {MAX_LENGTH} \
    --p-n-jobs {N_JOBS_EXTRACT} \
    --o-read-extraction-stats silva-{SILVA_VERSION}-v3v4-341f-806r-stats.qza \
    --o-reads silva-{SILVA_VERSION}-v3v4-341f-806r-seqs.qza \
    --use-cache {QIIME_CACHE} \
    --verbose


In [ ]:
!qiime tools peek silva-{SILVA_VERSION}-v3v4-341f-806r-seqs.qza

!qiime tools validate \
    silva-{SILVA_VERSION}-v3v4-341f-806r-seqs.qza


## 7. Train Naive Bayes classifiers

The QIIME 2 `fit-classifier-naive-bayes` action trains a scikit-learn-based taxonomic classifier using the extracted V3–V4 reference reads and the corresponding SILVA taxonomy.

The classifier is serialized together with the scikit-learn model, so it is advisable to use it with the same compatible QIIME 2/scikit-learn environment in which it was created.


### 7.1 Genus-level classifier


In [ ]:
!qiime feature-classifier fit-classifier-naive-bayes \
    --i-reference-reads silva-{SILVA_VERSION}-v3v4-341f-806r-seqs.qza \
    --i-reference-taxonomy silva-{SILVA_VERSION}-ssu-nr99-tax.qza \
    --o-classifier silva-{SILVA_VERSION}-v3v4-341f-806r-nb-classifier.qza \
    --use-cache {QIIME_CACHE} \
    --verbose


In [ ]:
!qiime tools peek \
    silva-{SILVA_VERSION}-v3v4-341f-806r-nb-classifier.qza

!qiime tools validate \
    silva-{SILVA_VERSION}-v3v4-341f-806r-nb-classifier.qza


### 7.2 Classifier including species labels


In [ ]:
!qiime feature-classifier fit-classifier-naive-bayes \
    --i-reference-reads silva-{SILVA_VERSION}-v3v4-341f-806r-seqs.qza \
    --i-reference-taxonomy silva-{SILVA_VERSION}-ssu-nr99-species-tax.qza \
    --o-classifier silva-{SILVA_VERSION}-v3v4-341f-806r-nb-species-classifier.qza \
    --use-cache {QIIME_CACHE} \
    --verbose


In [ ]:
!qiime tools peek \
    silva-{SILVA_VERSION}-v3v4-341f-806r-nb-species-classifier.qza

!qiime tools validate \
    silva-{SILVA_VERSION}-v3v4-341f-806r-nb-species-classifier.qza


## 8. Classify the reference reads

This step applies the trained classifier back to the extracted SILVA V3–V4 reads.

The purpose is to compare the expected SILVA taxonomy with the taxonomy predicted by the classifier. Because the same sequences were used during training, this is an **internal consistency evaluation**, not an independent validation.

### Temporary storage

`classify-sklearn` uses Joblib for multiprocessing. With multiple workers, Joblib may create large temporary memory-mapped files. `JOBLIB_TEMP_FOLDER` was therefore configured in Section 1 to point to the large temporary filesystem.


### 8.1 Genus-level classification


In [ ]:
!qiime feature-classifier classify-sklearn \
    --i-classifier silva-{SILVA_VERSION}-v3v4-341f-806r-nb-classifier.qza \
    --i-reads silva-{SILVA_VERSION}-v3v4-341f-806r-seqs.qza \
    --p-n-jobs {N_JOBS_CLASSIFY} \
    --p-reads-per-batch {READS_PER_BATCH} \
    --o-classification silva-{SILVA_VERSION}-v3v4-341f-806r-nb-observed-taxonomy.qza \
    --use-cache {QIIME_CACHE} \
    --verbose


In [ ]:
!qiime tools peek \
    silva-{SILVA_VERSION}-v3v4-341f-806r-nb-observed-taxonomy.qza

!qiime tools validate \
    silva-{SILVA_VERSION}-v3v4-341f-806r-nb-observed-taxonomy.qza


### 8.2 Classification including species labels


In [ ]:
!qiime feature-classifier classify-sklearn \
    --i-classifier silva-{SILVA_VERSION}-v3v4-341f-806r-nb-species-classifier.qza \
    --i-reads silva-{SILVA_VERSION}-v3v4-341f-806r-seqs.qza \
    --p-n-jobs {N_JOBS_CLASSIFY} \
    --p-reads-per-batch {READS_PER_BATCH} \
    --o-classification silva-{SILVA_VERSION}-v3v4-341f-806r-nb-species-observed-taxonomy.qza \
    --use-cache {QIIME_CACHE} \
    --verbose


In [ ]:
!qiime tools peek \
    silva-{SILVA_VERSION}-v3v4-341f-806r-nb-species-observed-taxonomy.qza

!qiime tools validate \
    silva-{SILVA_VERSION}-v3v4-341f-806r-nb-species-observed-taxonomy.qza


## 9. Evaluate expected versus observed taxonomy

RESCRIPt `evaluate-classifications` compares the expected reference taxonomy with the taxonomy predicted by the classifier.

The generated `.qzv` files summarize agreement across taxonomic ranks. Because this evaluation uses the training reference sequences, the results should be reported as **training-set/internal consistency metrics**.

For an unbiased estimate of generalization performance, an independent test set or an appropriate cross-validation/hold-out strategy would be required.


### 9.1 Genus-level evaluation


In [ ]:
!qiime rescript evaluate-classifications \
    --i-expected-taxonomies silva-{SILVA_VERSION}-ssu-nr99-tax.qza \
    --i-observed-taxonomies silva-{SILVA_VERSION}-v3v4-341f-806r-nb-observed-taxonomy.qza \
    --p-labels "SILVA {SILVA_VERSION} V3-V4 341F-806R" \
    --o-evaluation silva-{SILVA_VERSION}-v3v4-341f-806r-nb-evaluation.qzv \
    --use-cache {QIIME_CACHE} \
    --verbose


### 9.2 Species-label evaluation


In [ ]:
!qiime rescript evaluate-classifications \
    --i-expected-taxonomies silva-{SILVA_VERSION}-ssu-nr99-species-tax.qza \
    --i-observed-taxonomies silva-{SILVA_VERSION}-v3v4-341f-806r-nb-species-observed-taxonomy.qza \
    --p-labels "SILVA {SILVA_VERSION} V3-V4 341F-806R species labels" \
    --o-evaluation silva-{SILVA_VERSION}-v3v4-341f-806r-nb-species-evaluation.qzv \
    --use-cache {QIIME_CACHE} \
    --verbose


## 10. Inspect final artifacts

On a headless server, `.qzv` files are usually downloaded and opened with **QIIME 2 View**:

https://view.qiime2.org/

The commands below only verify that the final artifacts exist and can be interpreted by QIIME 2.


In [ ]:
!qiime tools peek \
    silva-{SILVA_VERSION}-v3v4-341f-806r-nb-evaluation.qzv

!qiime tools validate \
    silva-{SILVA_VERSION}-v3v4-341f-806r-nb-evaluation.qzv

!qiime tools peek \
    silva-{SILVA_VERSION}-v3v4-341f-806r-nb-species-evaluation.qzv

!qiime tools validate \
    silva-{SILVA_VERSION}-v3v4-341f-806r-nb-species-evaluation.qzv


## 11. Notes for reproducibility

When distributing or reporting this classifier, record at least:

- SILVA release: **138.2**;
- reference collection: **SSU Ref NR99**;
- target region: **16S V3–V4**;
- primers: **341F / 806R**;
- amplicon length filter: **350–550 bp**;
- QIIME 2 version;
- RESCRIPt version;
- q2-feature-classifier version;
- scikit-learn version;
- whether taxonomy was limited to genus or included species labels.

The `.qza` and `.qzv` artifacts preserve QIIME 2 provenance internally, but the notebook provides a human-readable record of the analytical decisions.


## 12. References

**Bokulich NA, Kaehler BD, Rideout JR, et al.** Optimizing taxonomic classification of marker-gene amplicon sequences with QIIME 2's q2-feature-classifier plugin. *Microbiome*. 2018;6:90.  
https://doi.org/10.1186/s40168-018-0470-z

**Quast C, Pruesse E, Yilmaz P, et al.** The SILVA ribosomal RNA gene database project: improved data processing and web-based tools. *Nucleic Acids Research*. 2013;41:D590–D596.  
https://doi.org/10.1093/nar/gks1219

**Robeson MS II, O'Rourke DR, Kaehler BD, et al.** RESCRIPt: Reproducible sequence taxonomy reference database management. *PLoS Computational Biology*. 2021;17:e1009581.  
https://doi.org/10.1371/journal.pcbi.1009581

**Yilmaz P, Parfrey LW, Yarza P, et al.** The SILVA and “All-species Living Tree Project (LTP)” taxonomic frameworks. *Nucleic Acids Research*. 2014;42:D643–D648.  
https://doi.org/10.1093/nar/gkt1209
